# Weight Initialization & Training Dynamics

Companion notebook for the [Weight Initialization lesson](https://ml-viz.vercel.app/courses/neural-networks/06-weight-initialization).

We propagate a signal through a deep network and watch initialization decide whether activations
**vanish, explode, or stay stable** — then verify that **He initialization** preserves variance
through a ReLU stack where naive init fails. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — Signal variance across depth depends on init scale

Push a random input through 30 linear+ReLU layers and track the activation standard deviation per
layer. Too-large init explodes it; too-small vanishes it; the right scale keeps it ~constant.

In [ ]:
def propagate(scale, depth=30, width=256, seed=0):
    r = np.random.default_rng(seed)
    h = r.normal(size=width)
    stds = [h.std()]
    for _ in range(depth):
        W = r.normal(size=(width, width)) * scale
        h = np.maximum(W @ h, 0)            # linear + ReLU
        stds.append(h.std())
    return stds

fig, ax = plt.subplots(figsize=(8, 4.2))
for scale, name, c in [(0.02, 'too small (vanish)', '#fb7185'),
                       (0.20, 'too large (explode)', '#eab308'),
                       (np.sqrt(2/256), 'He init (stable)', '#2dd4bf')]:
    ax.plot(propagate(scale), label=f'{name}, scale={scale:.3f}', color=c)
ax.set_yscale('log'); ax.set_xlabel('layer'); ax.set_ylabel('activation std (log)')
ax.set_title('Initialization scale decides whether the signal survives depth')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2 — He initialization preserves variance through ReLU

He init sets Var(W) = 2/n_in. The factor of 2 compensates for ReLU zeroing half its inputs. We
measure the per-layer variance ratio — it stays near 1 for He, but decays for the naive 1/n_in.

In [ ]:
def he_std(n_in):     return np.sqrt(2.0 / n_in)
def xavier_std(n_in, n_out): return np.sqrt(2.0 / (n_in + n_out))

width = 512
for name, scale in [('He (2/n)', he_std(width)), ('naive (1/n)', np.sqrt(1.0/width))]:
    stds = propagate(scale, depth=20, width=width, seed=3)
    print(f'{name:12s}: activation std at layer 1 = {stds[1]:.3f}, at layer 20 = {stds[20]:.3f}')
print('\nHe keeps the std roughly flat; the naive 1/n scale lets it decay with depth.')

## 3 — Learning-rate warmup

Warmup ramps the LR up over the first steps (when weights/optimizer stats are unreliable), then
decays it. We plot a linear-warmup + cosine-decay schedule.

In [ ]:
def lr_schedule(step, base=1e-3, warmup=200, total=2000):
    if step < warmup:
        return base * step / warmup                              # linear warmup
    prog = (step - warmup) / (total - warmup)
    return base * 0.5 * (1 + np.cos(np.pi * prog))               # cosine decay

steps = np.arange(2000)
lrs = [lr_schedule(s) for s in steps]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(steps, lrs, color='#818cf8')
ax.axvline(200, ls='--', color='#888', label='end of warmup')
ax.set_xlabel('training step'); ax.set_ylabel('learning rate')
ax.set_title('Linear warmup then cosine decay'); ax.legend(facecolor='#1a1d27', edgecolor='#444')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## ✏️ Your turn

**Exercise.** Implement `he_init_std(n_in)` (= √(2/n_in)) and `warmup_lr(step, base, warmup)` — the
linear warmup phase that returns `base·step/warmup` while ramping up and `base` once `step ≥ warmup`.

In [ ]:
def he_init_std(n_in):
    # TODO(you): standard deviation for He initialization
    return ...

def warmup_lr(step, base=1e-3, warmup=200):
    # TODO(you): base*step/warmup while step < warmup, else base
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(he_init_std(2), 1.0)                  # sqrt(2/2) = 1
assert np.isclose(he_init_std(512), np.sqrt(2/512))
assert warmup_lr(0) == 0.0                               # LR starts at 0
assert np.isclose(warmup_lr(100, 1e-3, 200), 5e-4)      # halfway through warmup
assert warmup_lr(500, 1e-3, 200) == 1e-3               # past warmup -> base LR
print('\u2713 He init scale and warmup schedule are correct')

<details>
<summary>Solution</summary>

```python
def he_init_std(n_in):
    return np.sqrt(2.0 / n_in)

def warmup_lr(step, base=1e-3, warmup=200):
    return base * step / warmup if step < warmup else base
```

He's factor of 2 exactly cancels the variance ReLU loses by zeroing half its inputs, keeping signal
stable through depth. Warmup protects the fragile first steps, when weights and adaptive-optimizer
statistics haven't settled.

</details>